# Video → npy TimeSformer → ép thành 8576 → so độ giống với npy Baidu (Colab, v3.9)

**Chạy lần lượt cell 1 → 9 từ trên xuống.** Nếu một cell báo lỗi thì đọc log ngay trong cell đó, sửa xong rồi mới chạy cell tiếp theo, hoặc bấm `Runtime → Run all`. Không có bước train model bình luận.

| Bước | Việc làm |
|---|---|
| 1–4 | Kiểm tra GPU, cấu hình, mount Drive, lấy code + cài thư viện |
| 5 | Giải nén video, tải npy Baidu gốc (8576 chiều) của đúng các trận có video |
| 6 | **Video → npy TimeSformer 768 chiều**: model Baidu đã fine-tune trên SoccerNet |
| 7 | **Ép 768 → 8576**: học trên các trận train, chọn tham số trên valid |
| 8 | **Bảng độ giống với npy Baidu gốc** trên các trận test (không dùng để học) |
| 9 | Xem thử npy vừa tạo |

**Cần có trên Drive** (`MyDrive/soccer_project/`):
- `dataset/Dataset.zip`: video `1_224p.mkv`, `2_224p.mkv`.
- `weights/*.pdparams`: weights TimeSformer, tải bằng trình duyệt từ link OneDrive trong README của [baidu-research/Soccernet-features](https://github.com/baidu-research/Soccernet-features).

Chọn GPU: `Runtime → Change runtime type → T4 GPU`.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Cấu hình

`DRIVE_ROOT` là thư mục dự án trên Drive của **người đang chạy**. Khi chạy bằng tài khoản khác: chủ Drive chia sẻ thư mục `soccer_project`, người chạy bấm chuột phải → **Sắp xếp → Thêm lối tắt vào Drive**. Cell 3 cũng tự tìm thư mục này.

In [ ]:
GITHUB_REPO = "https://github.com/ThuLinh3009/soccer_project.git"
GIT_BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soccer_project"
WORK_DIR = "/content/soccernet-caption-deltalab"   # code
DATA = f"{WORK_DIR}/dataset/caption-2024"          # npy Baidu + nhãn + npy TimeSformer
VIDEO_DIR = "/content/videos_raw"                  # video giải nén
TS_BATCH = 32                                      # T4: 32 (fp16). Nếu hết VRAM thì giảm xuống 16

## 3. Mount Google Drive, tìm video và weights

In [ ]:
from google.colab import drive
import os, glob
drive.mount("/content/drive", force_remount=True)

if not os.path.isdir(DRIVE_ROOT):
    found = [p for p in glob.glob("/content/drive/MyDrive/**/soccer_project", recursive=True)
             + glob.glob("/content/drive/Shareddrives/*/soccer_project") if os.path.isdir(p)]
    assert found, f"Không thấy {DRIVE_ROOT}: thêm lối tắt tới thư mục soccer_project hoặc sửa DRIVE_ROOT ở cell 2"
    DRIVE_ROOT = found[0]

VIDEO_ZIP = f"{DRIVE_ROOT}/dataset/Dataset.zip"
TS_DRIVE_DIR = f"{DRIVE_ROOT}/timesformer768"                 # sao lưu npy 768 chiều
MAPPER = f"{DRIVE_ROOT}/sn-models/mapper_timesformer.pt"      # phép ép + _report.json
weights = sorted(glob.glob(f"{DRIVE_ROOT}/**/*.pdparams", recursive=True))
TS_WEIGHTS = weights[0] if weights else None

print("DRIVE_ROOT :", DRIVE_ROOT)
print("video zip  :", VIDEO_ZIP, "✅" if os.path.exists(VIDEO_ZIP) else "❌ KHÔNG THẤY")
print("weights    :", TS_WEIGHTS or "❌ KHÔNG THẤY file .pdparams")
assert os.path.exists(VIDEO_ZIP) and TS_WEIGHTS

## 4. Lấy code từ GitHub và cài thư viện

In [ ]:
if os.path.isdir(f"{WORK_DIR}/.git"):
    !git -C {WORK_DIR} fetch -q origin && git -C {WORK_DIR} reset -q --hard origin/{GIT_BRANCH}
else:
    !git clone -q -b {GIT_BRANCH} {GITHUB_REPO} {WORK_DIR}
%cd {WORK_DIR}
!git log --oneline -1
!grep -m1 "^## v" CHANGELOG.md
!pip install -q SoccerNet av

## 5. Giải nén video và tải npy Baidu gốc

- Giải nén video 224p từ `Dataset.zip`.
- Tải `1_/2_baidu_soccer_embeddings.npy` (npy gốc dùng để so sánh) và nhãn của **đúng các trận có video** từ HuggingFace.
- Mất vài phút, mỗi phiên Colab chỉ cần chạy 1 lần.

In [ ]:
import zipfile
from huggingface_hub import hf_hub_download
from SoccerNet.utils import getListGames

if not glob.glob(f"{VIDEO_DIR}/**/[12]_224p.mkv", recursive=True):
    with zipfile.ZipFile(VIDEO_ZIP) as z:
        z.extractall(VIDEO_DIR, [m for m in z.namelist() if m.endswith("_224p.mkv") and "__MACOSX" not in m])
videos = sorted(p for p in glob.glob(f"{VIDEO_DIR}/**/[12]_224p.mkv", recursive=True) if "__MACOSX" not in p)
VIDEO_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(videos[0]))))  # chứa <giải>/<mùa>/<trận>
VIDEO_GAMES = sorted({os.path.relpath(os.path.dirname(v), VIDEO_ROOT) for v in videos})
print(len(videos), "video,", len(VIDEO_GAMES), "trận")

for g in VIDEO_GAMES:
    for f, repo, rev in [("Labels-caption.json", "SoccerNet/SN-Labels", "main"),
                         ("1_baidu_soccer_embeddings.npy", "SoccerNet/SN-Features", "baidu-soccer-embeddings"),
                         ("2_baidu_soccer_embeddings.npy", "SoccerNet/SN-Features", "baidu-soccer-embeddings")]:
        if not os.path.exists(f"{DATA}/{g}/{f}"):
            hf_hub_download(repo, f"{g}/{f}", repo_type="dataset", revision=rev, local_dir=DATA)

split_of = {g: s for s in ["train", "valid", "test", "challenge"] for g in getListGames([s], task="caption")}
for g in VIDEO_GAMES:
    print(f"{split_of.get(g, '?'):6} {g}")

## 6. Video → npy TimeSformer 768 chiều

- Mỗi hiệp cho ra `1_timesformer768.npy` / `2_timesformer768.npy`, mỗi giây một dòng, cùng số dòng với npy Baidu.
- Ước tính trên T4: khoảng 5–10 phút mỗi hiệp, 20 hiệp mất khoảng 2–3 giờ.
- npy được **sao lưu lên Drive** (`timesformer768/`). Nếu phiên Colab bị ngắt, chạy lại từ cell 1: cell này chép lại các hiệp đã xong và chỉ trích phần còn thiếu.

In [ ]:
import shutil, subprocess
def run(cmd):
    # chạy lệnh, in log trực tiếp; báo lỗi ngay tại cell này nếu lệnh thất bại
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}): xem log ngay phía trên")

def sync(src_root, dst_root):
    for f in glob.glob(f"{src_root}/**/*_timesformer768.npy", recursive=True):
        dst = os.path.join(dst_root, os.path.relpath(f, src_root))
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copyfile(f, dst)

if os.path.isdir(TS_DRIVE_DIR):
    sync(TS_DRIVE_DIR, DATA)   # các hiệp đã trích ở phiên trước
try:
    run(["python", "timesformer_extract.py", "--weights", TS_WEIGHTS, "--video_root", VIDEO_ROOT,
         "--out_root", DATA, "--batch_size", str(TS_BATCH)])
finally:
    sync(DATA, TS_DRIVE_DIR)   # lưu cả các hiệp đã xong nếu bị lỗi giữa chừng

missing = [f"{g}/{h}" for g in VIDEO_GAMES for h in (1, 2)
           if not os.path.exists(f"{DATA}/{g}/{h}_timesformer768.npy")]
print(2 * len(VIDEO_GAMES) - len(missing), "/", 2 * len(VIDEO_GAMES), "hiệp đã có npy 768 chiều (đã lưu lên Drive)")
assert not missing, f"Còn thiếu: {missing}. Chạy lại cell này"

## 7. Ép 768 → 8576

Phép ép là một ma trận phải **học** từ các cặp "TimeSformer giây t ↔ Baidu giây t". Vì vậy cell này chạy **sau** cell 6.

| Nhóm trận (split SoccerNet) | Dùng để |
|---|---|
| train | Học phép ép (ridge regression, 5 giây TimeSformer quanh giây t, tự tìm độ lệch thời gian) |
| valid | Chọn tham số |
| test | Đo độ giống với npy Baidu gốc. Các trận này không được dùng để học |

Thêm `--mlp` vào lệnh để thử thêm một mạng MLP nhỏ. Mạng này chỉ được giữ nếu tốt hơn ridge.

In [ ]:
for g in VIDEO_GAMES:
    ok = "".join("✅" if os.path.exists(f"{DATA}/{g}/{h}_{f}") else "❌"
                 for f in ("timesformer768.npy", "baidu_soccer_embeddings.npy") for h in (1, 2))
    print(f"{split_of.get(g, '?'):6} TS1 TS2 BD1 BD2 = {ok}  {g}")
run(["python", "feature_mapper.py", "train", "--soccernet_path", DATA, "--out", MAPPER,
     "--train_splits", "train", "--val_splits", "valid", "--test_splits", "test"])

## 8. Bảng độ giống với npy Baidu gốc

| Chỉ số | Ý nghĩa |
|---|---|
| `cosine` | Độ giống trung bình, 1 = giống hệt |
| `cosine_ge_0.95` | Tỉ lệ số giây đạt cosine ≥ 0,95 |
| `cosine_TPN`… | Độ giống theo từng khối trong 5 khối của Baidu |
| `pearson_r_mean`, `R2` | Mức ăn khớp theo từng chiều |
| `mốc baidu_next_second` | Baidu giây t so với giây t+1 của chính nó (khoảng 0,94) |
| `mốc baidu_random_second` | Hai giây ngẫu nhiên không liên quan (khoảng 0,67) |

In [ ]:
import json, pandas as pd
REPORT = MAPPER.replace(".pt", "_report.json")
assert os.path.exists(REPORT), "Chưa có bảng kết quả: cell 7 chưa chạy xong hoặc bị lỗi, xem log cell 7"
rep = json.load(open(REPORT))
print("offset:", rep["offset"], "| context:", rep["context"], "| lambda:", rep["lambda"], "| kind:", rep["kind"])
print("số trận:", {k: len(v) for k, v in rep["games"].items()})
display(pd.DataFrame({k: rep[k] for k in ("val", "test") if k in rep}).round(4))
display(pd.DataFrame({k.replace("reference_", "mốc_"): rep[k] for k in ("reference_val", "reference_test") if k in rep}).round(4))

## 9. Xem thử npy vừa tạo

In [ ]:
import numpy as np
g = VIDEO_GAMES[0]
ts = np.load(f"{DATA}/{g}/1_timesformer768.npy")
bd = np.load(f"{DATA}/{g}/1_baidu_soccer_embeddings.npy", mmap_mode="r")
print(g)
print("TimeSformer:", ts.shape, ts.dtype, "| Baidu gốc:", bd.shape, bd.dtype)
print(ts[:3, :6])